# Optical LUT example

Install the repository into this notebook's Python environment first:
`python -m pip install -e /path/to/SizeDistMerge`.
The Python files live directly in `src/`; the public import remains `sizedistmerge`.
No private data or absolute machine-specific paths are included.

Read a completed table, convert bin edges between refractive indices, and
preserve each bin's number concentration. The source index must match the
instrument's actual calibration; the target here is illustrative.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sizedistmerge as sdm
from sizedistmerge import optical_diameter as od

lut = od.SigmaLUT(str(sdm.lut_path('pops')))
source_ri = 1.615 + 0.001j
target_ri = 1.45 + 0j
response_bins = 100
diameters = np.geomspace(150., 3000., 400)
raw = lut.sigma_curve(lut.Dg, source_ri.real, source_ri.imag)
response, inverse = od.make_monotone_sigma_interpolator(
    lut.Dg, raw, response_bins=response_bins)

fig, ax = plt.subplots(figsize=(4, 4))
ax.loglog(lut.Dg, raw, label='Mie', linewidth=1.5)
ax.loglog(diameters, response(diameters), '--', label='Monotone', linewidth=1.5)
ax.set(xlim=(150, 3000), xlabel='Diameter (nm)', ylabel='Scattering cross-section (µm²)')
ax.legend()
fig.tight_layout()

In [ ]:
edges = np.array([150., 200., 300., 500., 800.])
distribution = np.array([100., 60., 20., 5.])
converted_edges = od.convert_do_lut(
    edges, source_ri, target_ri, lut, response_bins=response_bins)
converted_distribution = sdm.remap_dndlog_by_edges(edges, converted_edges, distribution)
np.testing.assert_allclose(
    sdm.counts_from_dndlog(distribution, edges_nm=edges),
    sdm.counts_from_dndlog(converted_distribution, edges_nm=converted_edges))
print('Converted edges (nm):', converted_edges)